In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/Fraudulent_E-Commerce_Transaction_Data.csv')
print(f"Dataset loaded: {df.shape}")
print(df.head())

Dataset loaded: (1472952, 16)
                         Transaction ID                           Customer ID  \
0  15d2e414-8735-46fc-9e02-80b472b2580f  d1b87f62-51b2-493b-ad6a-77e0fe13e785   
1  0bfee1a0-6d5e-40da-a446-d04e73b1b177  37de64d5-e901-4a56-9ea0-af0c24c069cf   
2  e588eef4-b754-468e-9d90-d0e0abfc1af0  1bac88d6-4b22-409a-a06b-425119c57225   
3  4de46e52-60c3-49d9-be39-636681009789  2357c76e-9253-4ceb-b44e-ef4b71cb7d4d   
4  074a76de-fe2d-443e-a00c-f044cdb68e21  45071bc5-9588-43ea-8093-023caec8ea1c   

   Transaction Amount     Transaction Date Payment Method Product Category  \
0               58.09  2024-02-20 05:58:41  bank transfer      electronics   
1              389.96  2024-02-25 08:09:45     debit card      electronics   
2              134.19  2024-03-18 03:42:55         PayPal    home & garden   
3              226.17  2024-03-16 20:41:31  bank transfer         clothing   
4              121.53  2024-01-15 05:08:17  bank transfer         clothing   

   Quantity  C

In [2]:
# Check missing
print("Missing values:")
print(df.isnull().sum())

# Drop rows dengan missing (jika ada)
df = df.dropna()
print(f"Shape after dropna: {df.shape}")

Missing values:
Transaction ID        0
Customer ID           0
Transaction Amount    0
Transaction Date      0
Payment Method        0
Product Category      0
Quantity              0
Customer Age          0
Customer Location     0
Device Used           0
IP Address            0
Shipping Address      0
Billing Address       0
Is Fraudulent         0
Account Age Days      0
Transaction Hour      0
dtype: int64
Shape after dropna: (1472952, 16)


In [3]:
# Separate features dan target
X = df.drop('Is Fraudulent', axis=1)
y = df['Is Fraudulent']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts()}")

Features shape: (1472952, 15)
Target shape: (1472952,)
Target distribution:
Is Fraudulent
0    1399114
1      73838
Name: count, dtype: int64


In [4]:
# Identify categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

# Encode categorical
label_encoders = {}
X_encoded = X.copy()

for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le
    print(f"Encoded {col}: {len(le.classes_)} unique values")

print(f"\nShape after encoding: {X_encoded.shape}")

Categorical columns: ['Transaction ID', 'Customer ID', 'Transaction Date', 'Payment Method', 'Product Category', 'Customer Location', 'Device Used', 'IP Address', 'Shipping Address', 'Billing Address']
Encoded Transaction ID: 1472952 unique values
Encoded Customer ID: 1472952 unique values
Encoded Transaction Date: 1346684 unique values
Encoded Payment Method: 4 unique values
Encoded Product Category: 5 unique values
Encoded Customer Location: 99135 unique values
Encoded Device Used: 3 unique values
Encoded IP Address: 1472651 unique values
Encoded Shipping Address: 1472948 unique values
Encoded Billing Address: 1472949 unique values

Shape after encoding: (1472952, 15)


In [5]:
# Split 80-20
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test_shape: {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts()}")
print(f"y_test distribution:\n{y_test.value_counts()}")

X_train shape: (1178361, 15)
X_test_shape: (294591, 15)
y_train distribution:
Is Fraudulent
0    1119291
1      59070
Name: count, dtype: int64
y_test distribution:
Is Fraudulent
0    279823
1     14768
Name: count, dtype: int64


In [6]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_train_scaled mean:\n{X_train_scaled.mean()}")
print(f"X_train_scaled std:\n{X_train_scaled.std()}")

X_train_scaled shape: (1178361, 15)
X_train_scaled mean:
Transaction ID       -1.128802e-17
Customer ID          -1.172217e-16
Transaction Amount    7.713479e-17
Transaction Date      7.332388e-18
Payment Method       -1.061267e-17
Product Category      6.608797e-18
Quantity              7.216613e-17
Customer Age          2.891952e-16
Customer Location    -8.152457e-18
Device Used           8.451542e-17
IP Address           -3.116265e-17
Shipping Address     -1.257119e-16
Billing Address       8.007739e-18
Account Age Days     -5.306333e-18
Transaction Hour      6.710100e-17
dtype: float64
X_train_scaled std:
Transaction ID        1.0
Customer ID           1.0
Transaction Amount    1.0
Transaction Date      1.0
Payment Method        1.0
Product Category      1.0
Quantity              1.0
Customer Age          1.0
Customer Location     1.0
Device Used           1.0
IP Address            1.0
Shipping Address      1.0
Billing Address       1.0
Account Age Days      1.0
Transaction Hour   

In [7]:
# Create models directory if not exists
import os
os.makedirs('../models', exist_ok=True)

# Save preprocessed data
joblib.dump(X_train_scaled, '../models/X_train_scaled.joblib')
joblib.dump(X_test_scaled, '../models/X_test_scaled.joblib')
joblib.dump(y_train, '../models/y_train.joblib')
joblib.dump(y_test, '../models/y_test.joblib')
joblib.dump(scaler, '../models/scaler.joblib')
joblib.dump(label_encoders, '../models/label_encoders.joblib')

print("✓ All preprocessed data saved to models/")
print(f"  - X_train_scaled.joblib")
print(f"  - X_test_scaled.joblib")
print(f"  - y_train.joblib")
print(f"  - y_test.joblib")
print(f"  - scaler.joblib")
print(f"  - label_encoders.joblib")

✓ All preprocessed data saved to models/
  - X_train_scaled.joblib
  - X_test_scaled.joblib
  - y_train.joblib
  - y_test.joblib
  - scaler.joblib
  - label_encoders.joblib
